# 1.3 传播效应与衰减

**学习目标**
- 理解电磁波在大气中传播时的衰减机制
- 掌握相移（phase shift）和差分相移的物理意义
- 了解比衰减（specific attenuation）与雷达变量的关系
- 认识大气气体和降水对雷达信号的影响

**参考文献**：Ryzhkov & Zrnic, Chapter 1, pp. 86-120

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, Dropdown, FloatSlider
import ipywidgets as widgets
plt.rcParams['font.family'] = ['DejaVu Sans', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False

## 1. 理论背景

### 传播方程

雷达接收功率的传播方程考虑了路径衰减：

$$P_r = \frac{P_t G^2 \lambda^2 \sigma}{(4\pi)^3 R^4} e^{-\int_0^R 2\alpha(r) dr}$$

其中 $\alpha$ 是传播路径上的衰减系数（dB/km）。

### 大气气体衰减

氧气和水汽是主要的衰减源。S波段的衰减约为 0.01 dB/km，C波段约 0.05 dB/km，X波段约 0.1 dB/km。

In [ ]:
def gas_attenuation(wavelength, distance, humidity=50, pressure=1013.25, temperature=15):
    """
    计算大气气体衰减
    基于 ITU-R P.676-12 推荐模型
    
    wavelength: 波长 (m)
    distance: 传播距离 (km)
    humidity: 相对湿度 (%)
    pressure: 大气压 (hPa)
    temperature: 温度 (°C)
    """
    freq = 3e8 / wavelength  # 频率 Hz
    
    # 氧气衰减系数 (dB/km)
    o2_atten = (7.19e-3 * freq) / (freq**2 + 0.044) * (pressure / 1013.25) * ((288 / (273 + temperature))**0.5)
    
    # 水汽衰减系数 (dB/km)
    h2o_atten = (0.05 + 0.0021 * humidity) * (pressure / 1013.25) * ((288 / (273 + temperature))**0.5) * (freq / 22)**2
    
    total_atten = (o2_atten + h2o_atten) * distance
    return total_atten, o2_atten * distance, h2o_atten * distance

def rain_attenuation(R, wavelength, distance):
    """
    计算降雨衰减
    R: 降雨率 (mm/h)
    wavelength: 波长 (m)
    distance: 距离 (km)
    """
    # 经验公式：k 和 alpha 参数
    if wavelength < 0.04:  # X-band
        k = 0.00059 * R**0.79
        alpha = 1.0
    elif wavelength < 0.08:  # C-band
        k = 0.000187 * R**0.82
        alpha = 1.0
    else:  # S-band
        k = 0.000138 * R**0.85
        alpha = 1.0
    
    gamma = k * R**alpha  # 比衰减 dB/km
    total_atten = gamma * distance
    return total_atten, gamma

def differential_phase_shift(KDP, distance):
    """
    计算差分相移 ΦDP
    KDP: 比差分相移 (°/km)
    distance: 距离 (km)
    """
    return KDP * distance

## 2. 衰减随距离和降雨率的变化

In [ ]:
def plot_propagation_effects(wavelength_type='C-band', rainfall_rate=10.0, distance=100.0):
    """绘制传播效应随距离的变化"""
    
    wavelengths = {'X-band': 0.032, 'C-band': 0.054, 'S-band': 0.107}
    wl = wavelengths[wavelength_type]
    
    # 距离范围
    r = np.linspace(1, distance, 200)
    
    # 气体衰减
    gas_att = [gas_attenuation(wl, d)[0] for d in r]
    
    # 降雨衰减
    rain_att = [rain_attenuation(rainfall_rate, wl, d)[0] for d in r]
    
    # 总衰减
    total_att = [g + w for g, w in zip(gas_att, rain_att)]
    
    # 差分相移（假设KDP与降雨率成正比）
    KDP = 0.05 * rainfall_rate  # 经验关系
    phidp = [differential_phase_shift(KDP, d) for d in r]
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # 子图1: 累积衰减 vs 距离
    ax1 = axes[0, 0]
    ax1.plot(r, gas_att, 'b-', linewidth=2, label='气体衰减')
    ax1.plot(r, rain_att, 'r-', linewidth=2, label=f'降雨衰减 (R={rainfall_rate}mm/h)')
    ax1.plot(r, total_att, 'k-', linewidth=2.5, label='总衰减')
    ax1.set_xlabel('距离 (km)', fontsize=11)
    ax1.set_ylabel('累积衰减 (dB)', fontsize=11)
    ax1.set_title(f'{wavelength_type} (λ={wl*100:.1f} cm) 累积衰减', fontsize=12)
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 子图2: 比衰减 vs 降雨率
    ax2 = axes[0, 1]
    R_range = np.linspace(0.1, 100, 100)
    for band, wl_val in wavelengths.items():
        gamma_range = [rain_attenuation(R, wl_val, 1.0)[1] for R in R_range]
        ax2.semilogy(R_range, gamma_range, linewidth=2, label=band)
    ax2.set_xlabel('降雨率 R (mm/h)', fontsize=11)
    ax2.set_ylabel('比衰减 γ (dB/km)', fontsize=11)
    ax2.set_title('降雨比衰减 vs 降雨率', fontsize=12)
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # 子图3: 差分相移 vs 距离
    ax3 = axes[1, 0]
    ax3.plot(r, phidp, 'g-', linewidth=2)
    ax3.set_xlabel('距离 (km)', fontsize=11)
    ax3.set_ylabel('ΦDP (°)', fontsize=11)
    ax3.set_title(f'差分相移 (KDP={KDP:.2f}°/km)', fontsize=12)
    ax3.grid(True, alpha=0.3)
    
    # 子图4: 反射率衰减校正
    ax4 = axes[1, 1]
    Z_true = 40  # 真实反射率 dBZ
    Z_measured = [Z_true - att for att in total_att]
    ax4.plot(r, [Z_true]*len(r), 'k--', linewidth=2, label='真实 Z')
    ax4.plot(r, Z_measured, 'r-', linewidth=2, label='测量 Z (已衰减)')
    ax4.set_xlabel('距离 (km)', fontsize=11)
    ax4.set_ylabel('反射率 Z (dBZ)', fontsize=11)
    ax4.set_title('反射率衰减与校正', fontsize=12)
    ax4.legend()
    ax4.grid(True, alpha=0.3)
    ax4.set_ylim(0, 45)
    
    plt.tight_layout()
    plt.show()
    
    # 打印统计信息
    print(f"\n=== {wavelength_type} 传播效应统计 ===")
    print(f"距离: {distance:.0f} km")
    print(f"降雨率: {rainfall_rate:.1f} mm/h")
    print(f"气体衰减: {gas_att[-1]:.2f} dB")
    print(f"降雨衰减: {rain_att[-1]:.2f} dB")
    print(f"总衰减: {total_att[-1]:.2f} dB")
    print(f"差分相移 ΦDP: {phidp[-1]:.1f}°")
    print(f"反射率低估量: {total_att[-1]:.1f} dB")

In [ ]:
# 交互式传播效应演示
interact_prop = interact(plot_propagation_effects,
    wavelength_type=widgets.Dropdown(options=['X-band', 'C-band', 'S-band'],
                                   value='C-band', description='雷达波段'),
    rainfall_rate=widgets.FloatSlider(min=0.1, max=100, step=0.5, 
                                      value=10.0, description='降雨率 (mm/h)'),
    distance=widgets.FloatSlider(min=10, max=300, step=10, 
                                value=100.0, description='距离 (km)')
)

## 3. 相位变化与KDP

比差分相移 KDP 是极化雷达的重要变量，它不受衰减影响，可以用来估计降雨率。

In [ ]:
def plot_kdp_relationship():
    """展示KDP与降雨率的关系"""
    R_range = np.linspace(0.1, 100, 100)
    
    # 经验关系：不同波段的 KDP(R)
    kdp_x = 0.062 * R_range**0.92  # X-band
    kdp_c = 0.028 * R_range**0.93  # C-band
    kdp_s = 0.014 * R_range**0.95  # S-band
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # 子图1: KDP vs R
    ax1 = axes[0]
    ax1.plot(R_range, kdp_x, 'r-', linewidth=2, label='X-band')
    ax1.plot(R_range, kdp_c, 'g-', linewidth=2, label='C-band')
    ax1.plot(R_range, kdp_s, 'b-', linewidth=2, label='S-band')
    ax1.set_xlabel('降雨率 R (mm/h)', fontsize=11)
    ax1.set_ylabel('KDP (°/km)', fontsize=11)
    ax1.set_title('KDP 与降雨率的经验关系', fontsize=12)
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 子图2: R(KDP) 反演
    ax2 = axes[1]
    kdp_range = np.linspace(0.1, 5, 100)
    R_x = (kdp_range / 0.062) ** (1/0.92)
    R_c = (kdp_range / 0.028) ** (1/0.93)
    R_s = (kdp_range / 0.014) ** (1/0.95)
    ax2.plot(kdp_range, R_x, 'r-', linewidth=2, label='X-band')
    ax2.plot(kdp_range, R_c, 'g-', linewidth=2, label='C-band')
    ax2.plot(kdp_range, R_s, 'b-', linewidth=2, label='S-band')
    ax2.set_xlabel('KDP (°/km)', fontsize=11)
    ax2.set_ylabel('反演降雨率 R (mm/h)', fontsize=11)
    ax2.set_title('从 KDP 反演降雨率', fontsize=12)
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # 典型值表格
    print("\n=== KDP 与降雨率对照表 ===")
    print(f"{'R (mm/h)':<12} {'X-band KDP':<15} {'C-band KDP':<15} {'S-band KDP':<15}")
    print("-" * 57)
    for R in [1, 5, 10, 20, 50, 100]:
        print(f"{R:<12} {0.062*R**0.92:<15.2f} {0.028*R**0.93:<15.2f} {0.014*R**0.95:<15.2f}")

In [ ]:
plot_kdp_relationship()

## 练习题

1. **衰减估算**：C波段雷达探测100km外的降雨（20mm/h），总衰减约多少dB？

2. **波段选择**：为什么在暴雨中S波段雷达比X波段更适合远距离探测？

3. **KDP优势**：解释为什么KDP不受衰减影响，而ZDR可能受衰减影响？

4. **衰减校正**：如果测量Z为30dBZ，在C波段20mm/h降雨中传播50km后，真实Z是多少？

## 参考文献

- Ryzhkov, A.V., and D.S. Zrnic, 2019: *Radar Polarimetry for Weather Observations*, Chapter 1, Springer.
- Doviak, R.J., and D.S. Zrnic, 1993: *Doppler Radar and Weather Observations*, 2nd ed., Academic Press.